<a href="https://colab.research.google.com/github/col-a-guo/guo_chen_jang_ms_project/blob/main/BERTearnings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch transformers datasets torchmetrics scikit-learn numpy huggingface-hub pandas imblearn pytorch_metric_learning

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import zipfile
import os

FILE_PATH = '/content/drive/MyDrive/BERTearningsdata/'
print(os.listdir('/content/drive/MyDrive/BERTearningsdata/'))

['train_sept22_combined.csv', 'test_sept22_combined.csv', 'sept22_combined.csv', 'sept1_combined.csv', 'test_sept1_combined.csv', 'train_sept1_combined.csv', 'sept22_combined.gsheet', 'train_nov25_combined.csv', 'test_nov25_combined.csv', 'nov25_combined.csv', 'test_dec1_combined.csv', 'train_dec1_combined.csv', 'feature_importance_inc_token_bottleneckBERT.csv', 'feature_importance_inc_token_bottleneckBERT.png', 'feature_importance_by_label_bottleneckBERT_nov25.png', 'feature_importance_by_label_bottleneckBERT_nov25.csv', 'feature_importance_by_label_bottleneckBERT_dec1_v2.png', 'feature_importance_by_label_bottleneckBERT_dec1_v2.csv', 'test_dec13_combined.csv', 'dec13_combined.csv', 'train_dec13_combined.csv', 'text_importance_bottleneckBERT.txt', 'feature_importance_label_0_bottleneckBERT.png', 'feature_importance_label_1_bottleneckBERT.csv', 'feature_importance_label_2_bottleneckBERT.csv', 'feature_importance_label_1_bottleneckBERT.png', 'text_importance_bottleneckBERT.csv', 'featur

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset, Subset
from datasets import load_dataset
from collections import Counter
import torchmetrics
from sklearn.metrics import confusion_matrix, classification_report
import random
import numpy as np
from huggingface_hub import HfApi, login
from huggingface_hub import PyTorchModelHubMixin
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from torch.optim.lr_scheduler import SequentialLR, LinearLR, ExponentialLR
import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
HF_REPO_ID =  "https://huggingface.co/colaguo/bottleneckBERT-3"

def upload_to_huggingface(model, tokenizer, version, token, repo_id):
    """
    Uploads the trained model weights and tokenizer to Hugging Face Hub.

    Args:
        model:     trained BertClassifier instance
        tokenizer: the tokenizer used during training
        version:   model version string (used for local save path)
        token:     your HF write token
        repo_id:   your HF repo, e.g. "username/repo-name"
    """
    print(f"\nUploading model and tokenizer to Hugging Face:  https://huggingface.co/colaguo/bottleneckBERT-3")

    # Authenticate
    login(token=token)
    api = HfApi()

    # --- Save model weights locally first ---
    local_model_dir = f"hf_upload_{version}"
    os.makedirs(local_model_dir, exist_ok=True)

    model_save_path = os.path.join(local_model_dir, "pytorch_model.bin")
    torch.save(model.state_dict(), model_save_path)
    print(f"Model weights saved locally to: {model_save_path}")

    # --- Save tokenizer locally ---
    tokenizer.save_pretrained(local_model_dir)
    print(f"Tokenizer saved locally to: {local_model_dir}")

    # --- Upload everything in the directory to HF ---
    api.upload_folder(
        folder_path=local_model_dir,
        repo_id=repo_id,
        repo_type="model",
        token=token,
    )

    print(f"Upload complete! View your model at: https://huggingface.co/{repo_id}")

if __name__ == "__main__":

    seed_value = 1
    random.seed(seed_value)
    np.random.seed(seed_value)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)

    version_list = ["bottleneckBERT"]
    # Default hyperparameters
    default_lr = 2e-5 #initial learning rate
    target_lr = 9e-6 #Target after 10 epochs
    default_eps = 6.748313060587885e-08
    default_batch_size = 32
    num_epochs = 200
    patience = 3
    dropout_rate = 0.2  # Added dropout parameter

    # function to generate classification report for multi-class
    def generate_classification_report(model, dataloader, num_classes, epoch=None, version=None, split_name="Test"):
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad(): #run once, don't update gradients during reporting
            for batch in dataloader:
                input_ids, attention_mask, features, Bottid_encoded, labels = [t.to(device) for t in batch] #unpack Bottid because one hot encoded
                logits, _ = model(input_ids, attention_mask, features, Bottid_encoded)  # Get logits and embeddings
                preds = torch.argmax(logits, dim=1)  # multi-class prediction
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Convert to numpy arrays for sklearn functions
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)

        report = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(num_classes)], digits=4)

        cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
        cm_report = "\nConfusion Matrix:\n"
        cm_report += "            Predicted\n"
        cm_report += "           " + "    ".join(map(str, range(num_classes))) + "\n"
        cm_report += "Actual\n"
        for i, row in enumerate(cm):
            cm_report += f"      {i}   " + "    ".join(map(str, row)) + "\n"


        final_report = f"""
    Classification Report ({split_name}, Version: {version}, Epoch {epoch if epoch is not None else 'Final'}):\n
    {report}\n
    {cm_report}
    """


        print(final_report)
        with open("classification_report.txt", "a") as f:
            f.write(final_report + "\n")

        f1 = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(num_classes)], output_dict=True, zero_division=0)['weighted avg']['f1-score'] # Added zero_division

        return f1, all_preds, all_labels

    def create_test_sets(test_dataset, num_sets=10, subset_size=0.9):
        """
        Splits the test set into `num_sets` subsets, each containing `subset_size` proportion
        of the data for each label, for Monte Carlo cross validation (MCCV)

        Args:
            test_dataset: your test dataset
            num_sets: int, the number of test subsets to create.
            subset_size: 0<float<1, proportion of subset (e.g., 0.2 for 20%).

        Returns:
            test_sets: subsets of your test dataset
        """

        # Get indices of samples for each label
        label_0_indices = [i for i, item in enumerate(test_dataset) if item[-1] == 0] #item[-1] is label
        label_1_indices = [i for i, item in enumerate(test_dataset) if item[-1] == 1]
        label_2_indices = [i for i, item in enumerate(test_dataset) if item[-1] == 2]  # Added for label 2

        # Calculate the number of samples to select for each label in each subset
        num_label_0_samples = int(len(label_0_indices) * subset_size)
        num_label_1_samples = int(len(label_1_indices) * subset_size)
        num_label_2_samples = int(len(label_2_indices) * subset_size)  # Added for label 2

        test_sets = []
        for _ in range(num_sets):
            # Randomly select indices for each label
            subset_label_0_indices = random.sample(label_0_indices, num_label_0_samples)
            subset_label_1_indices = random.sample(label_1_indices, num_label_1_samples)
            subset_label_2_indices = random.sample(label_2_indices, num_label_2_samples)  # Added for label 2

            # Combine the indices
            subset_indices = subset_label_0_indices + subset_label_1_indices + subset_label_2_indices  # Added label_2
            random.shuffle(subset_indices)  # Shuffle the combined indices

            # Create a Subset from the selected indices
            subset = Subset(test_dataset, subset_indices)
            test_sets.append(subset)

        return test_sets

    def evaluate_on_multiple_test_sets(model, test_sets, num_classes=3, version=None):
        """
        Evaluates the model on multiple test sets and calculates the average performance and standard deviations.

        Args:
            model: The trained PyTorch model.
            test_sets: test subsets from create_testsets
            num_classes: int, for classification
            version: str, for recording versions

        Returns:
            results: dict containing the average classification report metrics and standard deviations.
        """

        all_reports = []
        all_preds = []
        all_labels = []

        #take test sets and get results
        for i, test_set in enumerate(test_sets):
            dataloader = DataLoader(test_set, batch_size=default_batch_size)
            f1, preds, labels = generate_classification_report(model, dataloader, num_classes, version=version, split_name=f"Test Set {i+1}")
            all_reports.append(classification_report(labels, preds, target_names=[str(i) for i in range(num_classes)], output_dict=True, zero_division=0))
            all_preds.extend(preds)
            all_labels.extend(labels)

        #create metrics across test sets: mean, stdev across precision/recall/f1/support
        metrics = {}
        for class_idx in range(num_classes):
            class_str = str(class_idx)
            metrics[f'precision_{class_str}'] = [report[class_str]['precision'] for report in all_reports]
            metrics[f'recall_{class_str}'] = [report[class_str]['recall'] for report in all_reports]
            metrics[f'f1-score_{class_str}'] = [report[class_str]['f1-score'] for report in all_reports]
            metrics[f'support_{class_str}'] = [report[class_str]['support'] for report in all_reports]

        metrics['macro_avg_precision'] = [report['macro avg']['precision'] for report in all_reports]
        metrics['macro_avg_recall'] = [report['macro avg']['recall'] for report in all_reports]
        metrics['macro_avg_f1-score'] = [report['macro avg']['f1-score'] for report in all_reports]
        metrics['macro_avg_support'] = [report['macro avg']['support'] for report in all_reports]

        metrics['weighted_avg_precision'] = [report['weighted avg']['precision'] for report in all_reports]
        metrics['weighted_avg_recall'] = [report['weighted avg']['recall'] for report in all_reports]
        metrics['weighted_avg_f1-score'] = [report['weighted avg']['f1-score'] for report in all_reports]
        metrics['weighted_avg_support'] = [report['weighted avg']['support'] for report in all_reports]

        results = {}
        for metric_name, values in metrics.items():
            results[metric_name + "_avg"] = np.mean(values)
            results[metric_name + "_std"] = np.std(values)

        #Print final report
        final_report = "Averaged performance across all test sets:\n"
        for metric_name, value in results.items():
            if "_avg" in metric_name:
                std_name = metric_name.replace("_avg", "_std")
            if std_name in results: # Check to make sure that we don't cause a key error
                final_report += f"{metric_name}: {value:.4f} +/- {results[std_name]:.4f}\n"

        print(final_report)
        with open("/content/drive/MyDrive/BERTearningsdata/classification_report.txt", "a") as f:
            f.write(final_report + "\n")


        return results


    #main model architecture with dropout, batch norm, and residual connections
    class BertClassifier(nn.Module, PyTorchModelHubMixin):
        def __init__(self, version, num_labels=3, freeze_bert=False, num_Bottid_categories=29, dropout=0.2):
            super(BertClassifier, self).__init__()

            if version == "bert-uncased":
                self.bert = AutoModel.from_pretrained('google-bert/bert-base-uncased')
            elif version == "businessBERT":
                self.bert = AutoModel.from_pretrained('pborchert/BusinessBERT')
            elif version == "bottleneckBERT":
                self.bert = AutoModel.from_pretrained('colaguo/bottleneckBERTsmall')
            else:
                raise ValueError(f"Invalid model version: {version}")

            self.version = version

            # First linear layer for key features with BatchNorm and Dropout
            self.linear_features = nn.Sequential(
                nn.Linear(13, 32),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

            # First linear layer for Bottids with BatchNorm and Dropout
            self.linear_Bottid = nn.Sequential(
                nn.Linear(num_Bottid_categories, 8),
                nn.BatchNorm1d(8),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

            # First linear layer for BERT output with BatchNorm and Dropout
            self.cls_head = nn.Sequential(
                nn.Linear(self.bert.config.hidden_size, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

            # Combined layer with BatchNorm and Dropout
            self.linear_combined_layer = nn.Sequential(
                nn.Linear(256 + 32 + 8, 32),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

            # Residual projection to match dimensions (296 -> 32)
            self.residual_projection = nn.Linear(256 + 32 + 8, 32)

            self.final_classifier = nn.Linear(32, num_labels)

            self.pooling = nn.AdaptiveAvgPool1d(1) # Global avg pool


            if freeze_bert:
                for param in self.bert.parameters():
                    param.requires_grad = False

        def forward(self, input_ids, attention_mask, features, Bottid_encoded):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            # Use [CLS] token (first token) instead of average pooling
            cls_output = outputs.last_hidden_state[:, 0, :]

            bert_output = self.cls_head(cls_output)

            linear_features_output = self.linear_features(features)
            Bottid_output = self.linear_Bottid(Bottid_encoded)

            combined_output = torch.cat((bert_output, linear_features_output, Bottid_output), dim=1)

            # Apply residual connection
            # Project the combined input to match the output dimension
            residual = self.residual_projection(combined_output)
            linear_layer_output = self.linear_combined_layer(combined_output)

            # Add residual connection
            linear_layer_output = linear_layer_output + residual

            # Return both logits and embeddings
            logits = self.final_classifier(linear_layer_output)
            return logits, linear_layer_output



    # Function to load the correct tokenizer
    def load_tokenizer(version):
        if version == "bert-uncased":
            return AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')
        elif version == "businessBERT":
            return AutoTokenizer.from_pretrained('pborchert/BusinessBERT')
        elif version == "bottleneckBERT":
            return AutoTokenizer.from_pretrained('colaguo/bottleneckBERTsmall')
        else:
            raise ValueError(f"Invalid model version: {version}")
#####START EDIT##########
    # Load dataset and preprocess
    ogpath = "dec13_combined_bonus.csv"

    # Load the combined dataset
    combined_df = pd.read_csv("/content/drive/MyDrive/BERTearningsdata/" + ogpath)

    # Split based on year
    train_df = combined_df[combined_df['year'] != 1].copy()
    test_df = combined_df[combined_df['year'] == 1].copy()

    # Save temporary train and test files
    train_df.to_csv("/content/drive/MyDrive/BERTearningsdata/temp_train.csv", index=False)
    test_df.to_csv("/content/drive/MyDrive/BERTearningsdata/temp_test.csv", index=False)

    # Load using load_dataset to maintain the proper structure
    dataset = load_dataset('csv', data_files={
        'train': "/content/drive/MyDrive/BERTearningsdata/temp_train.csv",
        'test': "/content/drive/MyDrive/BERTearningsdata/temp_test.csv"
    })

    train_df = pd.read_csv("/content/drive/MyDrive/BERTearningsdata/temp_train.csv")
    test_df = pd.read_csv("/content/drive/MyDrive/BERTearningsdata/temp_test.csv")

    test_df.loc[test_df['label'] > 2, 'label'] = 2  # Or remove rows, or re-assign as needed

    encoder = OneHotEncoder(handle_unknown='ignore')

    encoder.fit(train_df[['Bottid']])
    train_encoded = encoder.transform(train_df[['Bottid']]).toarray()
    test_encoded = encoder.transform(test_df[['Bottid']]).toarray()

    feature_names = [f"Bottid_{i}" for i in range(train_encoded.shape[1])]

    # create a temporary dataframe to store encoded values, with feature names
    train_encoded_df = pd.DataFrame(train_encoded, columns=feature_names)
    test_encoded_df = pd.DataFrame(test_encoded, columns=feature_names)

    train_df = pd.concat([train_df, train_encoded_df], axis=1)
    test_df = pd.concat([test_df, test_encoded_df], axis=1)

    # Remove the original Bottid column
    train_df = train_df.drop('Bottid', axis=1)
    test_df = test_df.drop('Bottid', axis=1)

    #Convert the dataframes back to HuggingFace datasets
    dataset['train'] = dataset['train'].from_pandas(train_df)
    dataset['test'] = dataset['test'].from_pandas(test_df)
#########END EDIT###########
    # Truncate dataset
    def truncate_dataset(dataset):
        k = round(len(dataset)*0.99)
        random_indices = random.sample(range(len(dataset)), k)
        return dataset.select(random_indices)

    dataset = {k: truncate_dataset(v) for k, v in dataset.items()}

    def tokenize_function(examples, tokenizer):
        return tokenizer(examples["paragraph"], padding="max_length", truncation=True, max_length=512)

    class CustomDataset(Dataset):
        def __init__(self, dataset, Bottid_categories=29):
            self.dataset = dataset
            self.Bottid_categories = Bottid_categories

        def __len__(self):
            return len(self.dataset)

        def __getitem__(self, idx):
            item = self.dataset[idx]
            input_ids = torch.tensor(item['input_ids'])
            attention_mask = torch.tensor(item['attention_mask'])
            label = torch.tensor(item['label'], dtype=torch.long)
            features = torch.tensor([item['year'], item['word_count'], item['scarcity'], item['nonuniform_progress'], item['performance_constraints'], item['user_heterogeneity'], item['cognitive'], item['external'], item['internal'], item['coordination'], item['transactional'], item['technical'], item['demand']], dtype=torch.float)

            # Extract the one-hot encoded Bottid features
            Bottid_encoded = torch.tensor([item[f"Bottid_{i}"] for i in range(self.Bottid_categories)], dtype=torch.float)

            return input_ids, attention_mask, features, Bottid_encoded, label

    # Training function - simple version without freezing
    def train_and_evaluate(model, train_dataloader, val_dataloader, optimizer, epochs, loss_fn, patience=4, num_classes=3, version=None, test_sets=None):
        model.to(device)
        best_f1 = 0.0
        patience_counter = 0
        best_epoch = 0
        output_dir = "model_output"
        best_model_state = None

        # Create the output directory if it doesn't exist
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        #main training loop
        for epoch in range(epochs):
            model.train()
            total_loss = 0

            for batch in train_dataloader:
                input_ids, attention_mask, features, Bottid_encoded, labels = [t.to(device) for t in batch]
                model.zero_grad()

                logits, embeddings = model(input_ids, attention_mask, features, Bottid_encoded)

                # Cross-entropy loss only
                loss = loss_fn(logits, labels)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                total_loss += loss.item()

            avg_train_loss = total_loss / len(train_dataloader)

            # Validation
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch in val_dataloader:
                    input_ids, attention_mask, features, Bottid_encoded, labels = [t.to(device) for t in batch]
                    logits, _ = model(input_ids, attention_mask, features, Bottid_encoded)
                    val_loss += loss_fn(logits, labels).item()

            avg_val_loss = val_loss / len(val_dataloader)
            print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")

            # Generate and save the classification report every epoch
            f1_score,_,_ = generate_classification_report(model, val_dataloader, num_classes, epoch=epoch+1, version=version, split_name="Val")

            # Early stopping based on F1 score
            if f1_score > best_f1:
                best_f1 = f1_score
                best_epoch = epoch + 1
                patience_counter = 0
                best_model_state = model.state_dict()
                print(f"New best F1 score: {best_f1:.4f} at epoch {epoch+1}.")

            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print("Early stopping triggered.")
                    break

        # Load best model weights, then save
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            model_filename = f"model_output/model_version_{version}.pth"
            torch.save(model.state_dict(), model_filename)
            print(f"Best model (version {version}) saved to {model_filename} with F1 {best_f1:.4f}")

            # Evaluate and report on multiple test sets when a new best model is found
            if test_sets is not None:
                print("Evaluating on multiple test sets...")
                evaluate_on_multiple_test_sets(model, test_sets, num_classes=num_classes, version=version)
                print("Evaluation on multiple test sets complete.")

        print(f"Training completed. Best F1 score: {best_f1:.4f} achieved at epoch {best_epoch}.")
        return best_f1

    # Main loop
    for version in version_list:
        print(f"\n----- Running with {version} -----")

        tokenizer = load_tokenizer(version)
        tokenized_datasets = {split: data.map(lambda examples: tokenize_function(examples, tokenizer), batched=True) for split, data in dataset.items()}
        train_dataset = tokenized_datasets["train"]
        test_dataset = tokenized_datasets["test"]

        num_Bottid_categories = train_encoded.shape[1]
        train_data = CustomDataset(train_dataset, Bottid_categories=num_Bottid_categories)
        test_data = CustomDataset(test_dataset, Bottid_categories=num_Bottid_categories)

        #Create test sets:
        test_sets = create_test_sets(test_data)

        #Use full dataset
        train_data_loader = DataLoader(train_data, batch_size=default_batch_size, shuffle=True)
        normalized_weights = torch.tensor([1.0, 2, 1.3])
        loss_fn = nn.CrossEntropyLoss(weight=normalized_weights.to(device), label_smoothing=0.05)

        # Initialize Model with dropout, BERT unfrozen
        model = BertClassifier(version, num_labels=3, num_Bottid_categories=num_Bottid_categories, dropout=dropout_rate, freeze_bert=False).to(device)

        train_dataloader = train_data_loader
        val_dataloader = DataLoader(test_data, batch_size=default_batch_size)

        # Simple optimizer without scheduler
        optimizer = torch.optim.AdamW(model.parameters(), lr=default_lr, eps=default_eps)

        # Train and evaluate
        train_and_evaluate(model, train_dataloader, val_dataloader, optimizer, epochs=num_epochs, loss_fn=loss_fn, num_classes=3, version=version, test_sets=test_sets)

        #Evaluate on multiple test sets only after training + finding good model
        evaluate_on_multiple_test_sets(model, test_sets, num_classes=3, version=version)
        val_dataloader = DataLoader(test_data, batch_size=default_batch_size)
        generate_classification_report(model, val_dataloader, num_classes=3, version=version)

        upload_to_huggingface(model, tokenizer, version, token=HF_TOKEN, repo_id=HF_REPO_ID)

Using device: cuda
